# Install & imports

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from docx import Document as DocxDocument
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import FAISS

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "document_processing.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from document_processing import extract_pdf_pages, format_citation

# Load datasets

In [ ]:
DOCS_DIR = PROJECT_ROOT / "documents"
pdf_path = DOCS_DIR / "Why-We-Use-Common-Paper-Standard-Agreements.pdf"
source_documents = []

if pdf_path.exists():
    pdf_pages = extract_pdf_pages(pdf_path.read_bytes(), pdf_path.name)
    source_documents.extend(
        Document(
            page_content=str(page["text"]),
            metadata={"source": page["source"], "page_number": page["page_number"]},
        )
        for page in pdf_pages
        if str(page["text"]).strip()
    )

docx_files = [
    "Common-Paper-Cloud-Service-Agreement-SLA-v2.1-cover-page.docx",
    "Common-Paper-DPA-v1.1-cover-page-and-standard-terms.docx",
]
for filename in docx_files:
    docx_path = DOCS_DIR / filename
    if docx_path.exists():
        word_doc = DocxDocument(docx_path)
        text = "\n".join(p.text for p in word_doc.paragraphs if p.text.strip())
        if text.strip():
            source_documents.append(
                Document(page_content=text, metadata={"source": filename})
            )

print(f"Loaded {len(source_documents)} source documents/pages")
for source in source_documents:
    print(f"- {format_citation(source.metadata)}: {len(source.page_content):,} characters")

# Combine & tag text

In [ ]:
print(f"Loaded text total: {sum(len(doc.page_content) for doc in source_documents):,} characters")
print("PDF pages retain page numbers; DOCX sources are labeled by filename.")

# Chunk text

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
chunks = text_splitter.split_documents(source_documents)
for chunk_id, chunk in enumerate(chunks, start=1):
    chunk.metadata["chunk_id"] = chunk_id
print(f"Created {len(chunks)} chunks")

# Create embeddings & FAISS store

In [ ]:
embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(chunks, embeddings)
print("FAISS vector store created")

# Save & load index (optional persistence)

In [ ]:
vector_store.save_local("faiss_index")

# Setup QA chain

In [ ]:
llm = ChatOllama(
    model="llama3",
    temperature=0,
    base_url="http://localhost:11434",
)

prompt_template = """
You are a helpful AI assistant. Answer based ONLY on the provided context.
If the answer is not in the context, say "I don't know based on the provided document."
Do not make up answers.

Context:
{context}

Question: {question}

Answer:
"""

print("Ollama client configured. Ensure Ollama is running and llama3 is available before asking questions.")

# Interactive Q&A

In [ ]:
def ask(question, k=3):
    docs = vector_store.similarity_search(question, k=k)
    context = "\n\n".join(doc.page_content for doc in docs)
    response = llm.invoke(
        prompt_template.format(context=context, question=question)
    )

    print("\n=== ANSWER ===\n")
    print(response.content)

    print("\n=== SOURCES ===")
    for doc in docs:
        print(f"\n[{format_citation(doc.metadata)} ? chunk {doc.metadata['chunk_id']}]\n")
        print(doc.page_content[:400])

# Test questions

In [ ]:
ask("What is this document about?")
ask("What are the key service levels mentioned?")